# FS End-to-End Evaluation — 3-class triage confusion matrix

The FS-matcher analog of `end_to_end_eval.ipynb`'s triage confusion matrix. That notebook reads
reports whose Stage-4.5 decision-maker is the **ML (GBT) matcher** (the current production
default: `ml_feeds_clustering=True`, `fs_feeds_clustering=False`). This notebook produces and
reads the same 3x3 matrix, but for a run where the **FS matcher's own `auto_merge` tier** feeds
Stage 5 clustering instead (`fs_feeds_clustering=True`, `ml_feeds_clustering=False`) — i.e. "what
would end-to-end routing look like if FS were the shipped decision-maker".

**Why not just read `models/fs/fs_model_<ts>.meta.json`?** Its
`test_metrics.confusion_tier_by_label` is a **2x3** matrix: true label is binary (`0`/`1`, from
FS's own internal train/test split in `fs_matcher/train.py`), not the 3-class gold taxonomy
(`non-match` / `ambiguous` / `match`). It also carries no pipeline context at all — no blocking
misses, no deterministic-rule rejects, no gate drops, no clustering. It only tells you how FS's
`classification_tier` compares to the raw match/non-match label on FS's own held-out split. The
real end-to-end 3x3 matrix is a different, richer computation
(`src.evaluation.triage.triage_evaluation`, driven by `src.evaluation.pipeline_eval.evaluate_run`
against a full `RunManifest` + gold labels with the `ambiguous_pair` column) — reused here
unmodified, just pointed at an FS-fed run.

**Leakage.** FS is gold-trained (`meta.json`'s `silver_labels_path` in this checkout actually
points at a gold labels file), so exactly like the ML case, only a **`strict`** holdout is
leakage-free once a gold-trained stage feeds clustering — see `src/evaluation/holdout.py`
(the single shared hash-bucket split every training notebook and eval CLI holds out on) and the
`leakage` note `pipeline_eval.py` attaches to every report.

**VM-only.** `data/gold_labels/`, `data/runs/`, and `data/evaluations/` are all VM-only PHI,
gitignored, and empty in a local checkout — this notebook only runs where the real data lives.


## 0. Setup

In [ ]:
import sys
from pathlib import Path


def _service_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError("could not locate empi-service/ (no ancestor has data/ + src/)")


SERVICE_ROOT = _service_root(Path.cwd())
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import Settings, settings
from src.evaluation.report_io import load_reports, triage_matrix, triage_report

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)

print("service root:", SERVICE_ROOT)
print("runs dir    :", settings.runs_dir)
print("eval dir    :", settings.evaluations_dir)


### Chart styling

Copied verbatim from `end_to_end_eval.ipynb` so the heatmap matches the ML notebook's look
exactly, for later side-by-side comparison.

In [ ]:
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]  # blue, orange, aqua
INK, MUTED, GRID_C = "#0b0b0b", "#52514e", "#e6e5e1"

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 9,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.edgecolor": GRID_C,
    "axes.labelcolor": MUTED,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID_C,
    "grid.linewidth": 0.6,
    "legend.frameon": False,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
})


## 1. Run the FS-fed pipeline + evaluation

Skip this (`RUN = False`) to only read a report already written by a previous run of this cell.

Runs the pipeline **in process** with a one-off `Settings` override
(`fs_feeds_clustering=True, ml_feeds_clustering=False`) rather than mutating environment
variables or the shared `settings` singleton — every other setting (including all output
directories) stays identical, so `evaluate_run`/`report_io` below keep reading through the
normal, unmodified `settings`. This mirrors how `scripts/evaluate_all.py::evaluate_real()` calls
`run_pipeline` directly, just with the FS/ML toggles flipped and a single `strict`-holdout gold
evaluation (not the full real+synthetic, both-holdout battery — this notebook only needs the one
FS-fed triage matrix).

Set `REUSE_FS_RUN_ID` to an existing FS-fed `run_id` to skip re-running the pipeline, which is the
slow, libpostal-dependent part.

In [ ]:
from datetime import datetime, timezone

from src.pipeline import run_pipeline
from src.evaluation.pipeline_eval import evaluate_run, load_manifest, write_report
from src.evaluation.holdout import (
    DEFAULT_GOLD_LABELS,
    GOLD_AMBIGUOUS_COL,
    GOLD_LABEL_COL,
    holdout_keys,
    load_gold_labels,
)

RUN = True                  # flip to False to only read an existing report
FS_SESSION_ID = None         # e.g. "fs_end_to_end_v1"; None -> UTC timestamp
REUSE_FS_RUN_ID = None       # an existing FS-fed run_id, to skip re-running the pipeline

GOLD_PATH = DEFAULT_GOLD_LABELS   # the single shared gold file every eval CLI defaults to

if RUN:
    session_id = FS_SESSION_ID or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    if REUSE_FS_RUN_ID:
        print(f"Reusing FS-fed run {REUSE_FS_RUN_ID}")
        manifest = load_manifest(REUSE_FS_RUN_ID, settings)
    else:
        fs_settings = Settings(fs_feeds_clustering=True, ml_feeds_clustering=False)
        run_id = f"{session_id}_fs"
        print(f"Running the pipeline with fs_feeds_clustering=True, "
              f"ml_feeds_clustering=False -> run_id {run_id}")
        manifest = run_pipeline(raw_input=fs_settings.raw_input, settings=fs_settings,
                                run_id=run_id)

    if not GOLD_PATH.exists():
        raise SystemExit(
            f"Gold labels not found at {GOLD_PATH} — VM-only PHI, run this where the data lives."
        )
    labels = load_gold_labels(GOLD_PATH)
    # The single shared hash-bucket holdout (src.evaluation.holdout) — the same ~20% of
    # pairs every training notebook and eval CLI holds out on, so this is leakage-free
    # regardless of which pairs FS's own train/test split happened to use.
    strict_holdout = holdout_keys(labels)

    report = evaluate_run(
        manifest, labels, GOLD_LABEL_COL,
        settings=settings,
        label_source="gold",
        holdout=strict_holdout,
        holdout_name="strict",
        ambiguous_col=GOLD_AMBIGUOUS_COL,
        session_id=session_id,
    )
    print(report.to_text())
    written = write_report(report, settings)
    print("\nWrote report ->", written)
    FS_SESSION_ID = session_id
else:
    print("Skipped — reading a stored report only. Set FS_SESSION_ID to the session you want.")


## 2. Load the report

In [ ]:
reports = load_reports()
fs_reports = [
    r for r in reports
    if r.get("label_source") == "gold"
    and (r.get("leakage") or {}).get("restriction") == "strict"
    and r.get("session_id") == FS_SESSION_ID
]
if not fs_reports:
    raise SystemExit(
        f"No gold/strict report found for session {FS_SESSION_ID!r}. "
        "Run section 1 with RUN = True first, or set FS_SESSION_ID to an existing session."
    )

report = fs_reports[0]
holdout = report["leakage"]["restriction"]
print(f"run {report['run_id']}  |  session {report['session_id']}  |  "
      f"{report['label_source']}  |  holdout: {holdout}")
print(f"evaluated {report['evaluated_utc']}  |  pipeline git sha: {report['git_sha']}")
print(f"{report['universe']['labeled_pairs']} labeled pairs, "
      f"{report['universe']['positives']} positive")
if report["leakage"].get("note"):
    print("\n!", report["leakage"]["note"])


### 2.1 Triage — the three-class decision

| gold class | should be routed to |
|---|---|
| non-match (neither flag) | `no_match` |
| ambiguous (`ambiguous_pair`) | `human_review` |
| match (`final_gold_label`) | `auto_merge` |

A pair carrying *both* flags counts as **ambiguous** — see `src/evaluation/triage.py`'s module
docstring. Rows are the expected route from the gold labels; columns are where this FS-fed run
actually routed each pair (`auto_merge` here means "in the same cluster", and clustering was built
from deterministic-rule edges union FS's `auto_merge` tier, since `fs_feeds_clustering=True` for
this run).

In [ ]:
triage_report(report)

#### Confusion matrix

Rows are where the label says the pair belonged, columns where the FS-fed pipeline sent it. The
diagonal is correct routing; below it the pipeline was too cautious (a match left in review),
above it too aggressive (a non-match merged).

In [ ]:
triage_matrix(report)          # raw counts; normalize=True for row %

In [ ]:
def plot_triage_matrix(report, title=None):
    """Confusion matrix -> annotated heatmap. Copied from `end_to_end_eval.ipynb`.

    Shaded by row share (per-class recall), not raw count — non-matches outnumber matches
    several to one, so a count-shaded grid is one dark corner and two invisible rows. Each cell
    still prints its absolute count alongside the share.
    """
    counts = triage_matrix(report)
    if counts.empty:
        print("This report has no triage block (written before the section existed).")
        return
    counts = counts.drop(columns="total")
    pct = triage_matrix(report, normalize=True)

    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    im = ax.imshow(pct.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=100)

    ax.set_xticks(range(len(counts.columns)), counts.columns)
    ax.set_yticks(range(len(counts.index)), counts.index)
    ax.set_xlabel("system routed to")
    ax.set_ylabel("expected route")
    ax.grid(visible=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    for r in range(counts.shape[0]):
        for c in range(counts.shape[1]):
            share = pct.iat[r, c]
            share = 0.0 if pd.isna(share) else float(share)
            ax.text(c, r, f"{counts.iat[r, c]:,}\n{share:.1f}%",
                    ha="center", va="center", fontsize=8.5,
                    color="#ffffff" if share > 65 else INK)
            if r == c:   # ring the diagonal: the cells that mean "routed right"
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False,
                                           edgecolor=SERIES_COLORS[2], linewidth=2))

    cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
    cb.set_label("share of the expected class (%)", color=MUTED, fontsize=8)
    cb.outline.set_visible(False)
    ax.set_title(title or f"FS-fed triage routing — {report['label_source']} / {holdout}",
                 color=INK, loc="left")
    plt.tight_layout()
    plt.show()


plot_triage_matrix(report)
